# Hamilton-Jacobi Equation

This notebook contains the programmatic verification for the **Hamilton-Jacobi Equation** entry from the THEORIA dataset.

**Entry ID:** hamilton_jacobi_equation  
**Required Library:** sympy 1.13.1

## Description
The Hamilton-Jacobi equation is a first-order partial differential equation for Hamilton's principal function S. When solved, S generates a canonical transformation making the new Hamiltonian vanish, rendering equations of motion trivial. The conjugate momenta are obtained as partial derivatives of S with respect to coordinates.

## Installation
First, let's install the required library:

In [ ]:
# Install required library with exact version
!pip install sympy==1.13.1

## Programmatic Verification

The following code verifies the derivation mathematically:

In [ ]:
import sympy as sp

# =====================================================
# Programmatic verification: Hamilton-Jacobi Equation
# Verifies each derivation step systematically
# Uses single degree of freedom; generalizes to N DOF.
# =====================================================

# Define symbols
t = sp.Symbol('t', real=True)

# Generalized coordinates and momenta (single degree of freedom)
q = sp.Symbol('q', real=True)
p = sp.Symbol('p', real=True)

# New coordinates and momenta
Q = sp.Symbol('Q', real=True)
P = sp.Symbol('P', real=True)

# ---------------------------
# Step 1: Hamilton's equations (starting point from dependency)
# ---------------------------
H = sp.Function('H')
H_qpt = H(q, p, t)

# Hamilton's equations: dot_q = dH/dp, dot_p = -dH/dq
dot_q = sp.diff(H_qpt, p)
dot_p = -sp.diff(H_qpt, q)
print(f'Step 1: dot_q = {dot_q}, dot_p = {dot_p}')

# ---------------------------
# Step 2: Type-2 generating function transformation
# ---------------------------
G_2 = sp.Function('G_2')
G_2_qPt = G_2(q, P, t)

# Transformation relations: p = dG_2/dq, Q = dG_2/dP
p_transform = sp.diff(G_2_qPt, q)
Q_transform = sp.diff(G_2_qPt, P)
print(f'Step 2: p = {p_transform}, Q = {Q_transform}')

# ---------------------------
# Step 3: New Hamiltonian K = H + dG_2/dt
# ---------------------------
K = sp.Function('K')
K_expr = H_qpt + sp.diff(G_2_qPt, t)
print(f'Step 3: K = H + dG_2/dt = {K_expr}')

# ---------------------------
# Step 4: Hamilton's equations in new variables
# ---------------------------
K_QPt = K(Q, P, t)
dot_Q = sp.diff(K_QPt, P)
dot_P = -sp.diff(K_QPt, Q)
print(f'Step 4: dot_Q = {dot_Q}, dot_P = {dot_P}')

# ---------------------------
# Step 5: Set K = 0 for trivial dynamics
# ---------------------------
# When K = 0, dot_Q = 0 and dot_P = 0
K_zero = sp.Integer(0)
dot_Q_trivial = sp.diff(K_zero, P)
dot_P_trivial = -sp.diff(K_zero, Q)
assert dot_Q_trivial == 0, 'Step 5: dot_Q should be 0 when K=0'
assert dot_P_trivial == 0, 'Step 5: dot_P should be 0 when K=0'
print('Step 5 verified: K=0 implies dot_Q=0, dot_P=0')

# ---------------------------
# Step 6: Q and P are constants when K=0
# ---------------------------
alpha = sp.Symbol('alpha', real=True)  # P = alpha = const
beta = sp.Symbol('beta', real=True)    # Q = beta = const
print(f'Step 6: Q = beta (const), P = alpha (const)')

# ---------------------------
# Step 7: Set S = G_2 (with P replaced by constant alpha)
# ---------------------------
S = sp.Function('S')
S_qat = S(q, alpha, t)
G_2_as_S = S_qat
print(f'Step 7: G_2(q, alpha, t) = S(q, alpha, t)')

# ---------------------------
# Step 8: Momentum relation p = dS/dq (proves eq2)
# ---------------------------
p_from_S = sp.diff(G_2_as_S, q)
dS_dq = sp.diff(S_qat, q)
# dG_2/dq = dS/dq
assert sp.simplify(p_from_S - dS_dq) == 0, 'Step 8: p = dS/dq verification failed'
print(f'Step 8 verified (eq2): p = dG_2/dq = dS/dq = {dS_dq}')

# ---------------------------
# Step 9: Apply K = 0 condition
# ---------------------------
# K = H + dG_2/dt = 0
dG_2_dt = sp.diff(G_2_as_S, t)
K_condition = H_qpt + dG_2_dt
print(f'Step 9: K = H + dG_2/dt = 0 gives: {K_condition} = 0')

# ---------------------------
# Step 10: dG_2/dt = dS/dt since G_2 = S
# ---------------------------
dS_dt = sp.diff(S_qat, t)
assert sp.simplify(dG_2_dt - dS_dt) == 0, 'Step 10: dG_2/dt = dS/dt verification failed'
print(f'Step 10 verified: dG_2/dt = dS/dt = {dS_dt}')

# ---------------------------
# Step 11: Hamilton-Jacobi equation (proves eq1)
# ---------------------------
# Substitute p = dS/dq into H(q, p, t) + dS/dt = 0
# This gives H(q, dS/dq, t) + dS/dt = 0
H_with_dSdq = H(q, dS_dq, t)
HJ_equation = H_with_dSdq + dS_dt
print(f'Step 11 (eq1): H(q, dS/dq, t) + dS/dt = {HJ_equation} = 0')

# ---------------------------
# Concrete verification: Free particle
# ---------------------------
print('\n--- Concrete verification: Free particle ---')

m = sp.Symbol('m', positive=True)
E = sp.Symbol('E', positive=True)

# Free particle Hamiltonian: H = p^2/(2m)
H_free = p**2 / (2*m)

# Ansatz: S = W(q) - E*t for time-independent H
# Here E plays the role of alpha (constant of integration)
W = sp.Function('W')(q)
S_free = W - E*t

# Step 8: p = dS/dq = dW/dq
p_free = sp.diff(S_free, q)
assert p_free == sp.diff(W, q), 'Free particle: p = dW/dq failed'
print(f'p = dS/dq = {p_free}')

# Step 11: H(q, dS/dq, t) + dS/dt = 0
H_substituted = H_free.subs(p, p_free)
dS_dt_free = sp.diff(S_free, t)
HJ_free = H_substituted + dS_dt_free
print(f'HJ equation: {sp.simplify(HJ_free)} = 0')

# Solve: (dW/dq)^2/(2m) - E = 0 => dW/dq = sqrt(2mE)
dW_dq_solution = sp.sqrt(2*m*E)
HJ_check = (dW_dq_solution**2)/(2*m) - E
assert sp.simplify(HJ_check) == 0, 'HJ equation verification failed'
print('Free particle HJ equation verified!')

print('\n=== All derivation steps verified ===')
print('eq1: H(q, dS/dq, t) + dS/dt = 0 (Hamilton-Jacobi equation)')
print('eq2: p_k = dS/dq_k (momentum from principal function)')


## Source

📖 **View this entry:** [theoria-dataset.org/entries.html?entry=hamilton_jacobi_equation.json](https://theoria-dataset.org/entries.html?entry=hamilton_jacobi_equation.json)

This verification code is part of the [THEORIA dataset](https://github.com/theoria-dataset/theoria-dataset), a curated collection of theoretical physics derivations with programmatic verification.

**License:** CC-BY 4.0